In [250]:
import pandas as pd
import pyodbc
import csv

In [251]:
# 1. Extract Data

In [252]:
source_data = pd.read_csv('C:/Users/AS72180/OneDrive - ETS/Documents/ETS Hiver2024/ProjectMTI820/Amazon Sale Report.csv', 
                          dtype={"fulfilled-by": str, "Unnamed: 22": str})
source_data.drop(columns=["Unnamed: 22"], inplace=True)
source_data.drop(columns=["index"], inplace=True)

In [253]:
# 2. Transform Data
# Clean data, handle missing values, etc.

In [254]:
# Define a dictionary to map old column names to new column names
column_mapping = {
    'Order ID' : 'Order_ID' ,
    'Sales Channel' : 'Sales_Channel' ,
    'ship-service-level' : 'ship_service_level', 
    'Courier Status' : 'Courier_Status', 
    'ship-city' : 'ship_city',
    'ship-state' : 'ship_state',
    'ship-postal-code' : 'ship_postal_code',
    'ship-country' : 'ship_country',  
    'promotion-ids' : 'promotion_ids', 
    'fulfilled-by': 'fulfilled_by',
}

# Rename the columns
source_data.rename(columns=column_mapping, inplace=True)

# Print the DataFrame to verify the changes
print(source_data.head())

              Order_ID      Date                        Status Fulfilment  \
0  405-8078784-5731545  04-30-22                     Cancelled   Merchant   
1  171-9198151-1101146  04-30-22  Shipped - Delivered to Buyer   Merchant   
2  404-0687676-7273146  04-30-22                       Shipped     Amazon   
3  403-9615377-8133951  04-30-22                     Cancelled   Merchant   
4  407-1069790-7240320  04-30-22                       Shipped     Amazon   

  Sales Channel  ship_service_level    Style              SKU       Category  \
0      Amazon.in           Standard   SET389   SET389-KR-NP-S            Set   
1      Amazon.in           Standard  JNE3781  JNE3781-KR-XXXL          kurta   
2      Amazon.in          Expedited  JNE3371    JNE3371-KR-XL          kurta   
3      Amazon.in           Standard    J0341       J0341-DR-L  Western Dress   
4      Amazon.in          Expedited  JNE3671  JNE3671-TU-XXXL            Top   

  Size  ... Qty currency  Amount    ship_city   ship_sta

In [255]:
# Check for null values
null_values = source_data.isnull()

# Count null values in each column
null_counts = null_values.sum()

# Check if there are any null values in the DataFrame
if null_counts.any():
    print("DataFrame contains null values.")
    # Print the count of null values in each column
    print("Null value counts per column:")
    print(null_counts)
else:
    print("DataFrame does not contain null values.")

DataFrame contains null values.
Null value counts per column:
Order_ID                  0
Date                      0
Status                    0
Fulfilment                0
Sales Channel             0
ship_service_level        0
Style                     0
SKU                       0
Category                  0
Size                      0
ASIN                      0
Courier_Status         6872
Qty                       0
currency               7795
Amount                 7795
ship_city                33
ship_state               33
ship_postal_code         33
ship_country             33
promotion_ids         49153
B2B                       0
fulfilled_by          89698
dtype: int64


In [256]:
# Select categorical columns
categorical_columns = source_data.select_dtypes(include=['object']).columns.tolist()

# Select numerical columns
numerical_columns = source_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

Categorical columns:
['Order_ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship_service_level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier_Status', 'currency', 'ship_city', 'ship_state', 'ship_country', 'promotion_ids', 'fulfilled_by']

Numerical columns:
['Qty', 'Amount', 'ship_postal_code']


In [257]:
print(source_data.columns)

Index(['Order_ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ',
       'ship_service_level', 'Style', 'SKU', 'Category', 'Size', 'ASIN',
       'Courier_Status', 'Qty', 'currency', 'Amount', 'ship_city',
       'ship_state', 'ship_postal_code', 'ship_country', 'promotion_ids',
       'B2B', 'fulfilled_by'],
      dtype='object')


In [258]:
# List of specific categorical columns with missing values
categorical_cols_with_missing = ['Courier_Status', 'currency', 'ship_city', 'ship_state', 'ship_country', 'promotion_ids', 'fulfilled_by']

# Fill missing values in the specified categorical columns with 'Unknown'
source_data[categorical_cols_with_missing] = source_data[categorical_cols_with_missing].fillna('Unknown')

# Check if there are any remaining NaN values in the specified categorical columns
null_values_categorical = source_data[categorical_cols_with_missing].isnull()
null_counts_categorical = null_values_categorical.sum()
print("Null value counts per specified categorical column after filling with 'Unknown':")
print(null_counts_categorical)


Null value counts per specified categorical column after filling with 'Unknown':
Courier_Status    0
currency          0
ship_city         0
ship_state        0
ship_country      0
promotion_ids     0
fulfilled_by      0
dtype: int64


In [259]:
source_data.isnull().sum()

Order_ID                 0
Date                     0
Status                   0
Fulfilment               0
Sales Channel            0
ship_service_level       0
Style                    0
SKU                      0
Category                 0
Size                     0
ASIN                     0
Courier_Status           0
Qty                      0
currency                 0
Amount                7795
ship_city                0
ship_state               0
ship_postal_code        33
ship_country             0
promotion_ids            0
B2B                      0
fulfilled_by             0
dtype: int64

In [260]:
# Iterate over each row in the DataFrame
for index, row in source_data.iterrows():
    # Check if the 'promotion_ids' value is 'Unknown'
    if row['promotion_ids'] == 'Unknown':
        # Skip this row and continue to the next iteration
        continue
    
    # Split the value in the 'promotion_ids' column by comma (',') delimiter and keep only the first part
    split_values = row['promotion_ids'].split(',')
    first_part = split_values[0]
    # Replace the original value with the modified one
    source_data.at[index, 'promotion_ids'] = first_part

In [261]:
source_data.to_csv('C:/Users/AS72180/OneDrive - ETS/Documents/ETS Hiver2024/ProjectMTI820/Modified_Amazon_Sale_Report.csv', index=False)

In [262]:
# 3. Load Data
# Connect to the target destination ( Azure SQL database)

In [263]:
import pyodbc
import csv

# Define your Azure SQL Database connection details
server = 'serversalemti820.database.windows.net'
database = 'sqldb-sales'
username = 'adminmti820'
password = 'MTI820@2024'
driver = '{ODBC Driver 18 for SQL Server}'

# Establish a connection to the database
conn = pyodbc.connect('DRIVER='+driver+';SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)

# Create a cursor object to execute SQL queries
cursor = conn.cursor()

# Define the table creation query
create_table_query = """
CREATE TABLE Sales (
    Order_ID NVARCHAR(255),
    Date DATE,
    Status NVARCHAR(255),
    Fulfilment NVARCHAR(255),
    Sales_Channel NVARCHAR(255),
    ship_service_level NVARCHAR(255),
    Style NVARCHAR(255),
    SKU NVARCHAR(255),
    Category NVARCHAR(255),
    Size NVARCHAR(255),
    ASIN NVARCHAR(255),
    CourierStatus NVARCHAR(255),
    Qty INT,
    Currency NVARCHAR(255),
    Amount FLOAT,
    ship_city NVARCHAR(255),
    ship_state NVARCHAR(255),
    ship_postal_code NVARCHAR(255),
    ship_country NVARCHAR(255), 
    promotion_ids NVARCHAR(255),
    B2B BIT,
    fulfilled_by NVARCHAR(255)
)
"""

# Drop the table if it exists
cursor.execute("DROP TABLE IF EXISTS Sales")

# Create the table
cursor.execute(create_table_query)

# Commit the transaction
conn.commit()

# Path to your CSV file
csv_file = 'C:/Users/AS72180/OneDrive - ETS/Documents/ETS Hiver2024/ProjectMTI820/Modified_Amazon_Sale_Report.csv'
# Open the CSV file and read its contents
with open(csv_file, 'r') as file:
    reader = csv.reader(file)
    next(reader)  # Skip the header row if it exists
    for row in reader:
        # Insert each row into the database
        cursor.execute("INSERT INTO Sales (Order_ID, Date, Status, Fulfilment, Sales_Channel, ship_service_level, Style, SKU, Category, Size, ASIN, CourierStatus, Qty, Currency, Amount, ship_city, ship_state, ship_postal_code, ship_country, promotion_ids, B2B, fulfilled_by) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", row)

# Commit the transaction
conn.commit()

# Close the cursor and connection
cursor.close()
conn.close()

print("Data imported successfully.")

Data imported successfully.
